# XiangqiMaster — Colab Training

GPU training for the four agents. Run cells top-to-bottom. Colab disconnects,
so **checkpoints are written to Google Drive every epoch** and training can be
resumed with `--resume`.

Pipeline: **IL (Agent 2 P1) → PPO fine-tune (Agent 2 P2)**, plus **PPO self-play
(Agent 1)** and **MCTS (Agent 3)**.

> Set **Runtime → Change runtime type → GPU** first.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Get the code

**Option A — clone from GitHub** (push this repo first, then set the URL).
For a private repo use a token URL: `https://<TOKEN>@github.com/<user>/<repo>.git`.

In [ ]:
REPO_URL = 'https://github.com/<your-user>/xiangqimaster.git'  # <-- EDIT
!git clone $REPO_URL xiangqimaster
%cd xiangqimaster

**Option B — mount Google Drive** (if you uploaded the repo as a folder there).
Skip this if you cloned above.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/xiangqimaster

## 2. Install dependencies

Colab ships a CUDA `torch`, so we install only the RL/data libs (avoids
reinstalling a CPU torch over the GPU one).

In [ ]:
!pip -q install 'gymnasium>=0.29' 'stable-baselines3>=2.2' 'sb3-contrib>=2.2' \
    'numpy>=1.24' 'pandas>=2.0' 'tqdm>=4.66'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Checkpoint directory on Drive

Mount Drive and point checkpoints there so a disconnect never loses progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/xiangqi_ckpts'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

## 4. Get the data & build splits

The CGLemon dpxq/WXF game files live on Google Drive. Download them into
`data/raw/`, then build the train/val/test splits (parses ~141k games, spot-
checks legality — a few minutes).

If you already built `data/splits/*.jsonl` and put them on Drive, copy them into
`data/splits/` instead and skip the pipeline call.

In [ ]:
!pip -q install gdown
import os; os.makedirs('data/raw', exist_ok=True)
# CGLemon chinese-chess-PGN — the two ICCS collections (folder IDs from its README):
!gdown --folder 1NScafJyq3qVG7HO77U_4rVQJxbyRKe0O -O data/raw/wxf   # 41,743 games
!gdown --folder 12Js9Ld6Yixq4RA96j1PeT2QTUJ0-z0OB -O data/raw/dpxq  # 99,813 games
!ls -R data/raw | head

In [ ]:
# Build reproducible splits (ICCS notation, spot-check 500 for a legality rate).
!python -m src.data.pipeline --raw data/raw --out data/splits --notation iccs --spot-check 500

## 5. Imitation Learning — Agent 2, Phase 1

Trains the Policy-Value net to imitate human moves. Checkpoint is written to
Drive **every epoch**; if Colab drops, re-run this cell with `--resume` to
continue from the last epoch's weights. Expect ~6–8 h for 30 epochs on a T4;
professional-imitation top-1 accuracy of ~40–55% is normal.

In [ ]:
IL_CKPT = f'{CKPT_DIR}/il_agent2_phase1.pt'
!python -m src.training.il_train --splits data/splits --epochs 30 \
    --device cuda --num-workers 2 --batch-size 512 --save $IL_CKPT
# If disconnected, re-run with --resume appended to continue:
# !python -m src.training.il_train --splits data/splits --epochs 30 \
#     --device cuda --num-workers 2 --batch-size 512 --save $IL_CKPT --resume

## 6. PPO fine-tune — Agent 2, Phase 2

Continues from the IL checkpoint: its shared body is transferred into the PPO
features extractor (policy/value heads start fresh), then self-play PPO refines
it. `--il-checkpoint` must match the model's channels/num_blocks (both default
to config).

In [ ]:
!python -m src.training.ppo_train --timesteps 500000 \
    --il-checkpoint $IL_CKPT --save $CKPT_DIR/ppo_agent2

## 7. PPO self-play — Agent 1 (from scratch)

Independent of the IL checkpoint — random-init self-play. 500k steps ≈ 15–20 h;
split across sessions (SB3 `.zip` checkpoints on Drive).

In [ ]:
!python -m src.training.ppo_train --timesteps 500000 \
    --save $CKPT_DIR/ppo_agent1

## 8. MCTS self-play — Agent 3

The most compute-heavy agent (AlphaZero-style). See `docs/07-TRAINING.md §4`
for the loop and hyperparameters; wire the entrypoint the same way once its
training script is finalised.

## 9. Run the experiments (thesis Elo numbers)

With the trained checkpoints on Drive, run the experiment scripts (already
coded) to produce the comparison tables — e.g. Experiment 1:

In [ ]:
# Example — adjust to each experiment's CLI / entrypoint:
# !python -m experiments.exp1_main_comparison
!ls experiments/ 2>/dev/null || echo 'see experiments/ in the repo'

---
### Disconnect survival checklist
- Checkpoints go to **Drive** (`CKPT_DIR`), not Colab's ephemeral disk.
- IL: re-run the cell with `--resume` to continue from the last epoch.
- PPO: reload the SB3 `.zip` with `MaskablePPO.load(...)` and keep training.
- Keep the browser tab active; Colab Free times out when idle.